In [ ]:
"""
Bluesky post fetcher.

This script collects public posts from the Bluesky social network
(https://bsky.social) for a fixed set of search queries and a fixed
date range, and writes the results to local JSONL files and a SQLite
database. 

------------------
The Bluesky search endpoint (`app.bsky.feed.searchPosts`) returns at most
~100 results per request and is paginated by an opaque cursor. To obtain
broad coverage of a long time range, the script issues independent
queries per (search term, UTC day, sort order) tuple, where sort order
alternates between "top" and "latest". Each tuple is paginated until the
server returns no further cursor or until a per-tuple page cap is hit.
The day-bucketed approach is needed because the endpoint's `since`/`until`
parameters do not by themselves yield exhaustive results across a long
window; bucketing per day, per query, per sort widens the effective
coverage at the cost of substantial overlap, which is removed by
deduplication on post URI.

"""

import os
import json
import time
import sqlite3
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, Iterable, List, Optional, Tuple

import requests
from tqdm.auto import tqdm

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Time helpers
# The Bluesky API expects ISO-8601 strings with a trailing "Z" for UTC; Python's datetime emits "+00:00"

UTC = timezone.utc

def iso_z(dt: datetime) -> str:
    dt = dt.astimezone(UTC).replace(microsecond=0)
    return dt.isoformat().replace("+00:00", "Z")

def parse_iso_z(s: str) -> datetime:
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    return datetime.fromisoformat(s).astimezone(UTC)

def day_range_utc(day: datetime) -> Tuple[datetime, datetime]:
#Return the [start, end) UTC interval covering the calendar day of *day*.
    d0 = day.astimezone(UTC)
    start = datetime(d0.year, d0.month, d0.day, tzinfo=UTC)
    end = start + timedelta(days=1)
    return start, end

def iter_days(start_inclusive: datetime, end_exclusive: datetime) -> Iterable[datetime]:
    start = start_inclusive.astimezone(UTC)
    end = end_exclusive.astimezone(UTC)
    cur = datetime(start.year, start.month, start.day, tzinfo=UTC)
    while cur < end:
        yield cur
        cur += timedelta(days=1)

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Configuration
# API host, date range, search terms, output paths, and pagination/throttling

SEARCH_XRPCS = ["https://bsky.social/xrpc"]

# Corpus time window.
START_DT = datetime(2024, 12, 1, tzinfo=UTC)
END_DT   = datetime(2026, 2, 1, tzinfo=UTC)

# Search terms used to retrieve posts. 
QUERIES = [
    "ai art", "genai", "gen ai", "generative ai", "human art",
    "ai artist", "creative ai", "image generator", "ai slop",
    "ai generated", "ai artwork", "aiartist", "diffusion model",
    "gpt image", "stable diffusion", "midjourney",
]

# Output paths.
POSTS_OUT        = "posts_unique.jsonl"      # one row per unique post
MATCHES_OUT      = "post_matches.jsonl"      # one row per (post, query, day, sort)
CONSOLIDATED_OUT = "posts_consolidated.jsonl"  # joined export, written by export_consolidated()

# Resumable-state and dedup-index paths.
STATE_PATH  = "state.json"
SQLITE_PATH = "dedupe.sqlite"

# Pagination and throttling parameters.
# API max is ~100 results per page / delay between successful page requests / hard cap on pages per (query, day, sort) tuple
PAGE_LIMIT = 100
SLEEP_S = 0.25
MAX_PAGES_PER_DAY_PER_SORT = 50

# Authentication. Credentials read from .env
USE_AUTH = True
BSKY_HANDLE = os.getenv("BSKY_HANDLE") or os.getenv("USERNAME")
BSKY_APP_PASSWORD = os.getenv("BSKY_APP_PASSWORD") or os.getenv("PASSWORD")

# Session-token lifecycle parameters.
# refresh every 45 minutes / cooldown after 401
TOKEN_REFRESH_SECONDS = 45 * 60
AUTH_COOLDOWN_SECONDS = 60

# Day-window handling.
RECORD_OUT_OF_DAY_MATCHES = True
DAY_WINDOW_GRACE_SECONDS = 0

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# State
# tracks progress so that fetcher is resumable.

def load_state(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "queries": {},
        "_accessJwt": None,
        "_token_created_at": None,
        "_auth_cooldown_until": None,
    }

def save_state(path: str, state: Dict[str, Any]) -> None:
    tmp = path + ".tmp"

    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

    # windows-safe atomic replace with retry.
    last_err: Optional[PermissionError] = None
    for attempt in range(5):
        try:
            os.replace(tmp, path)
            return
        except PermissionError as e:
            last_err = e
            time.sleep(0.2 * (attempt + 1))

    raise PermissionError(
        f"Could not replace state file {path!r} after 5 attempts: {last_err}"
    )


def ensure_state_slot(state: Dict[str, Any], q: str, day_key: str, sort: str) -> Dict[str, Any]:
    qobj = state.setdefault("queries", {}).setdefault(q, {})
    dobj = qobj.setdefault(day_key, {})
    return dobj.setdefault(sort, {"cursor": None, "done": False, "pages": 0})

state = load_state(STATE_PATH)

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# SQLite store
# for deduplication and canonical post storage

def init_db(conn: sqlite3.Connection) -> None:
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    conn.execute("PRAGMA temp_store=MEMORY;")

    # dedup index, URI is the primary key.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS seen_posts (
            uri TEXT PRIMARY KEY,
            createdAt TEXT
        )
    """)

    # canonical normalised-post store, keyed by URI.
    conn.execute("""
        CREATE TABLE IF NOT EXISTS posts_json (
            uri TEXT PRIMARY KEY,
            json TEXT NOT NULL
        )
    """)

    # match table
    conn.execute("""
        CREATE TABLE IF NOT EXISTS post_matches (
            uri TEXT NOT NULL,
            matched_query TEXT NOT NULL,
            day_bucket TEXT NOT NULL,
            sort TEXT NOT NULL,
            createdAt TEXT,
            in_day_window INTEGER NOT NULL,
            PRIMARY KEY (uri, matched_query, day_bucket, sort)
        )
    """)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_matches_query ON post_matches(matched_query)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_matches_day ON post_matches(day_bucket)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_matches_sort ON post_matches(sort)")

    conn.commit()

def mark_seen(conn: sqlite3.Connection, uri: str, created_at: Optional[str]) -> bool:
    try:
        conn.execute("INSERT INTO seen_posts(uri, createdAt) VALUES(?, ?)", (uri, created_at))
        return True
    except sqlite3.IntegrityError:
        return False

def upsert_post_json(conn: sqlite3.Connection, uri: str, obj: Dict[str, Any]) -> None:
    conn.execute(
        "INSERT OR IGNORE INTO posts_json(uri, json) VALUES(?, ?)",
        (uri, json.dumps(obj, ensure_ascii=False)),
    )

def mark_match(
    conn: sqlite3.Connection,
    uri: str,
    matched_query: str,
    day_bucket: str,
    sort: str,
    created_at: Optional[str],
    in_day_window: bool,
) -> bool:
    try:
        conn.execute(
            (uri, matched_query, day_bucket, sort, created_at, 1 if in_day_window else 0),
        )
        return True
    except sqlite3.IntegrityError:
        return False

conn = sqlite3.connect(SQLITE_PATH)
init_db(conn)

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# HTTP layer
# handles authentication, retries with backoff on transient errors, and host fallback

session = requests.Session()
session.headers.update({"User-Agent": "post_fetcher/1.0"})

def safe_get(
    url: str,
    params: Dict[str, Any],
    headers: Optional[Dict[str, str]] = None,
    max_retries: int = 6
) -> requests.Response:
    backoff = 1.0
    for _ in range(max_retries):
        try:
            r = session.get(url, params=params, headers=headers, timeout=30)

            if r.status_code == 429:
                reset = r.headers.get("RateLimit-Reset") or r.headers.get("ratelimit-reset")
                if reset and str(reset).isdigit():
                    wait = max(0, int(reset) - int(time.time()))
                    time.sleep(wait)
                else:
                    time.sleep(backoff)
                    backoff = min(backoff * 2, 60)
                continue

            if r.status_code in (409, 413, 502, 503, 504):
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)
                continue

            return r

        except requests.RequestException:
            # DNS, TCP, TLS, read timeout, etc. Treat as transient.
            time.sleep(backoff)
            backoff = min(backoff * 2, 60)

    raise RuntimeError(f"Failed after {max_retries} tries: {url}")

def create_session_bearer(handle: str, app_password: str) -> str:
    url = "https://bsky.social/xrpc/com.atproto.server.createSession"
    r = session.post(url, json={"identifier": handle, "password": app_password}, timeout=30)
    r.raise_for_status()
    return r.json()["accessJwt"]

def get_valid_headers() -> Optional[dict]:
    if not USE_AUTH:
        return None
    if not (BSKY_HANDLE and BSKY_APP_PASSWORD):
        raise RuntimeError("USE_AUTH=True but BSKY_HANDLE/BSKY_APP_PASSWORD missing.")

    now = time.time()

    # honour any active cooldown from a previous 401.
    cooldown_until = state.get("_auth_cooldown_until")
    if cooldown_until and now < cooldown_until:
        time.sleep(int(cooldown_until - now))
        now = time.time()

    token = state.get("_accessJwt")
    created_at = state.get("_token_created_at")

    needs_refresh = (
        not token
        or created_at is None
        or (now - float(created_at) > TOKEN_REFRESH_SECONDS)
    )

    if needs_refresh:
        token = create_session_bearer(BSKY_HANDLE, BSKY_APP_PASSWORD)
        state["_accessJwt"] = token
        state["_token_created_at"] = now
        state["_auth_cooldown_until"] = None
        save_state(STATE_PATH, state)

    return {"Authorization": f"Bearer {state['_accessJwt']}"}

def search_posts_page(q: str, sort: str, cursor: Optional[str], since: str, until: str) -> Dict[str, Any]:
 # fetch one page of search results for a single (query, sort, day) tuple
    params = {"q": q, "limit": PAGE_LIMIT, "sort": sort, "since": since, "until": until}
    if cursor:
        params["cursor"] = cursor

    last_err = None
    for base in SEARCH_XRPCS:
        url = f"{base}/app.bsky.feed.searchPosts"
        headers = get_valid_headers()
        r = safe_get(url, params=params, headers=headers)

        if r.status_code == 401 and USE_AUTH:
            state["_token_created_at"] = 0
            state["_auth_cooldown_until"] = time.time() + AUTH_COOLDOWN_SECONDS
            save_state(STATE_PATH, state)
            time.sleep(AUTH_COOLDOWN_SECONDS)
            headers = get_valid_headers()
            r = safe_get(url, params=params, headers=headers)

        if r.status_code == 403:
            last_err = r.text[:200]
            continue

        if r.status_code == 400:
            raise RuntimeError(f"400 from {base} params={params} body={r.text[:200]}")

        r.raise_for_status()
        return r.json()

    raise RuntimeError(f"All search hosts failed/403. Last error: {last_err}")

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Post extraction and normalisation
# Bluesky posts are returned by the API as nested objects, normalisation functions flatten them

def extract_labels_from_post_view(pv: Dict[str, Any]) -> List[Dict[str, Any]]:
    labs = pv.get("labels") or []
    out = []
    for lab in labs:
        if isinstance(lab, dict):
            out.append({
                "src": lab.get("src"),
                "uri": lab.get("uri"),
                "cid": lab.get("cid"),
                "val": lab.get("val"),
                "neg": lab.get("neg"),
                "cts": lab.get("cts"),
            })
    return out

def extract_self_labels_from_record(record: Dict[str, Any]) -> List[str]:
    labs = record.get("labels") or {}
    if isinstance(labs, dict) and labs.get("$type") == "com.atproto.label.defs#selfLabels":
        vals = labs.get("values") or []
        return [v.get("val") for v in vals if isinstance(v, dict) and v.get("val")]
    return []

# label values that indicate adult or sensitive media
MEDIA_LABEL_VALUES = {"porn", "sexual", "graphic-media", "nudity"}

def derive_media_labels(row: Dict[str, Any]) -> List[str]:
    vals = []
    for lab in row.get("labels") or []:
        v = (lab or {}).get("val")
        if v in MEDIA_LABEL_VALUES:
            vals.append(v)
    for v in row.get("self_labels") or []:
        if v in MEDIA_LABEL_VALUES:
            vals.append(v)
    return sorted(set(vals))

def extract_image_alts(record: Dict[str, Any]) -> List[str]:
  # return alt-text strings from an `app.bsky.embed.images` embed
    emb = record.get("embed") or {}
    if emb.get("$type") != "app.bsky.embed.images":
        return []
    alts = []
    for img in (emb.get("images") or []):
        alt = img.get("alt")
        if alt:
            alts.append(alt)
    return alts

def extract_facets_summary(record: Dict[str, Any]) -> Dict[str, List[str]]:
  # extracts mentions, hashtags, and links 
    facets = record.get("facets") or []
    mentions, hashtags, links = [], [], []

    for facet in facets:
        for feat in (facet.get("features") or []):
            t = feat.get("$type")
            if t == "app.bsky.richtext.facet#mention":
                did = feat.get("did")
                if did:
                    mentions.append(did)
            elif t == "app.bsky.richtext.facet#tag":
                tag = feat.get("tag")
                if tag:
                    hashtags.append(tag)
            elif t == "app.bsky.richtext.facet#link":
                uri = feat.get("uri")
                if uri:
                    links.append(uri)

    def uniq(xs):
        # order-preserving deduplication, standard `set()` would lose order.
        seen = set()
        out = []
        for x in xs:
            if x not in seen:
                seen.add(x)
                out.append(x)
        return out

    return {"mentions": uniq(mentions), "hashtags": uniq(hashtags), "links": uniq(links)}

def extract_reply_refs(record: Dict[str, Any]) -> Dict[str, Optional[str]]:
  # return parent and root reply URIs, or none if the post is not a reply
    rep = record.get("reply") or {}
    parent = (rep.get("parent") or {}).get("uri")
    root = (rep.get("root") or {}).get("uri")
    return {"reply_parent_uri": parent, "reply_root_uri": root}

def extract_embed_summary(record: Dict[str, Any]) -> Dict[str, Any]:
    emb = record.get("embed") or {}
    et = emb.get("$type")

    out = {"embed_type": et, "external_uri": None, "quote_uri": None, "image_count": 0}

    if et == "app.bsky.embed.images":
        imgs = emb.get("images") or []
        out["image_count"] = len(imgs)
    elif et == "app.bsky.embed.external":
        ext = emb.get("external") or {}
        out["external_uri"] = ext.get("uri")
    elif et == "app.bsky.embed.record":
        rec = emb.get("record") or {}
        out["quote_uri"] = rec.get("uri")
    elif et == "app.bsky.embed.recordWithMedia":
        rec = emb.get("record") or {}
        out["quote_uri"] = (rec.get("record") or {}).get("uri") or rec.get("uri")
        media = emb.get("media") or {}
        if media.get("$type") == "app.bsky.embed.images":
            imgs = media.get("images") or []
            out["image_count"] = len(imgs)

    return out

def normalize_post_view(pv: Dict[str, Any]) -> Dict[str, Any]:
 # single row per post, flattening nested structures and extracting relevant fields for analysis  
    record = pv.get("record") or {}
    author = pv.get("author") or {}
    langs = record.get("langs")

    facets_sum = extract_facets_summary(record)
    reply_refs = extract_reply_refs(record)
    embed_sum  = extract_embed_summary(record)

    row = {
        "uri": pv.get("uri"),
        "cid": pv.get("cid"),
        "createdAt": record.get("createdAt"),
        "indexedAt": pv.get("indexedAt"),
        "text": record.get("text", "") or "",
        "langs": langs if isinstance(langs, list) else None,

        "author_did": author.get("did"),
        "author_handle": author.get("handle"),
        "author_displayName": author.get("displayName"),

        "replyCount": pv.get("replyCount"),
        "repostCount": pv.get("repostCount"),
        "likeCount": pv.get("likeCount"),
        "quoteCount": pv.get("quoteCount"),

        "image_alts": extract_image_alts(record),

        "labels": extract_labels_from_post_view(pv),
        "self_labels": extract_self_labels_from_record(record),

        "facets": facets_sum,
        **reply_refs,
        **embed_sum,
    }

    row["media_labels"] = derive_media_labels(row)
    return row

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Main fetch loop

def run_daily_fetch():
 # runs the full fetch over all (query, day, sort) tuples, with pagination and checkpointing
  
    # deduplicate the query list while preserving order
    seen_q = set()
    queries = [q for q in QUERIES if not (q in seen_q or seen_q.add(q))]

    pending_ops = 0
    def commit_if_needed(force: bool = False):
        nonlocal pending_ops
        if force or pending_ops >= 500:
            conn.commit()
            pending_ops = 0

    days = list(iter_days(START_DT, END_DT))
    sorts = ("top", "latest")

    with open(POSTS_OUT, "a", encoding="utf-8") as f_posts, \
         open(MATCHES_OUT, "a", encoding="utf-8") as f_matches:

        for q in tqdm(queries, desc="Queries", leave=True):
            for day_start in tqdm(days, desc=f"Days ({q})", leave=False):
                day_key = day_start.strftime("%Y-%m-%d")
                since_dt, until_dt = day_range_utc(day_start)

                grace = timedelta(seconds=int(DAY_WINDOW_GRACE_SECONDS or 0))
                since_dt_grace = since_dt - grace
                until_dt_grace = until_dt + grace

                since_iso = iso_z(since_dt)
                until_iso = iso_z(until_dt)

                for sort in sorts:
                    slot = ensure_state_slot(state, q, day_key, sort)
                    # skip tuples already completed in a previous run.
                    if slot.get("done"):
                        continue

                    cursor = slot.get("cursor")
                    pages = int(slot.get("pages", 0))

                    while True:
                        # per-tuple page cap
                        if MAX_PAGES_PER_DAY_PER_SORT is not None and pages >= MAX_PAGES_PER_DAY_PER_SORT:
                            slot["done"] = True
                            slot["cursor"] = None
                            save_state(STATE_PATH, state)
                            break

                        pages += 1
                        slot["pages"] = pages

                        data = search_posts_page(q=q, sort=sort, cursor=cursor, since=since_iso, until=until_iso)
                        posts = data.get("posts", []) or []
                        cursor = data.get("cursor")
                        slot["cursor"] = cursor

                        # Empty page on the first request, or empty page terminate this tuple
                        if not posts:
                            slot["done"] = True
                            slot["cursor"] = None
                            save_state(STATE_PATH, state)
                            break

                        new_posts_written = 0
                        new_matches_written = 0

                        for pv in posts:
                            row = normalize_post_view(pv)
                            uri = row.get("uri")
                            created_at = row.get("createdAt")
                            # Skip malformed posts that lack a URI or creation timestamp
                            if not uri or not created_at:
                                continue

                            try:
                                cdt = parse_iso_z(created_at)
                            except Exception:
                                # unparseable timestamp, drop the post
                                continue

                            # Global window filter - timestamp that falls outsite the corpus window gets discarded
                            if not (START_DT <= cdt < END_DT):
                                continue

                            in_day_window = (since_dt_grace <= cdt < until_dt_grace)

                            # record the match
                            if in_day_window or RECORD_OUT_OF_DAY_MATCHES:
                                match_inserted = mark_match(conn, uri, q, day_key, sort, created_at, in_day_window)
                                pending_ops += 1
                                if match_inserted:
                                    f_matches.write(json.dumps({
                                        "uri": uri,
                                        "createdAt": created_at,
                                        "matched_query": q,
                                        "day_bucket": day_key,
                                        "sort": sort,
                                        "in_day_window": bool(in_day_window),
                                        "since": since_iso,
                                        "until": until_iso,
                                        "grace_seconds": int(DAY_WINDOW_GRACE_SECONDS or 0),
                                    }, ensure_ascii=False) + "\n")
                                    new_matches_written += 1

                            # record the post itself
                            post_inserted = mark_seen(conn, uri, created_at)
                            pending_ops += 1
                            if post_inserted:
                                f_posts.write(json.dumps(row, ensure_ascii=False) + "\n")
                                new_posts_written += 1
                                upsert_post_json(conn, uri, row)
                                pending_ops += 1

                        commit_if_needed()
                        save_state(STATE_PATH, state)

                        tqdm.write(
                            f"{day_key} | {sort:6} | q='{q}' | page={pages} | "
                            f"new_posts={new_posts_written} new_matches={new_matches_written} | "
                            f"cursor={'end' if not cursor else '...'}"
                        )

                        # end of pagination for this tuple
                        if not cursor:
                            slot["done"] = True
                            slot["cursor"] = None
                            save_state(STATE_PATH, state)
                            break

                        time.sleep(SLEEP_S)

    commit_if_needed(force=True)

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Resume reporting
# a quick summary of where the fetcher is currently at

def resume_summary(
    state: Dict[str, Any],
    queries: List[str],
    start_dt: datetime,
    end_dt: datetime,
    sorts: Tuple[str, ...] = ("top", "latest"),
    show_incomplete_only: bool = True,
    max_lines_per_query: int = 10,
) -> None:
    seen = set()
    queries = [q for q in queries if not (q in seen or seen.add(q))]

    days = list(iter_days(start_dt, end_dt))
    day_keys = [d.strftime("%Y-%m-%d") for d in days]

    total_segments = len(queries) * len(day_keys) * len(sorts)
    done_segments = partial_segments = not_started_segments = 0

    q_stats = {}
    next_work = {}

    qobj_all = state.get("queries", {})

    for q in queries:
        qobj = qobj_all.get(q, {})
        q_done = q_partial = q_not_started = 0
        first_incomplete = None

        for day in day_keys:
            dobj = qobj.get(day, {})
            for s in sorts:
                slot = dobj.get(s)
                if not slot:
                    q_not_started += 1
                    if first_incomplete is None:
                        first_incomplete = (day, s, "not-started", 0, None)
                    continue

                pages = int(slot.get("pages", 0) or 0)
                done = bool(slot.get("done", False))
                cursor = slot.get("cursor")

                if done:
                    q_done += 1
                else:
                    # A segment with pages or a cursor is in progress;
                    # one with neither is effectively not started.
                    if pages > 0 or cursor:
                        q_partial += 1
                        if first_incomplete is None:
                            first_incomplete = (day, s, "in-progress", pages, cursor)
                    else:
                        q_not_started += 1
                        if first_incomplete is None:
                            first_incomplete = (day, s, "not-started", pages, cursor)

        q_stats[q] = (q_done, q_partial, q_not_started)
        if first_incomplete:
            next_work[q] = first_incomplete

        done_segments += q_done
        partial_segments += q_partial
        not_started_segments += q_not_started

    pct = (done_segments / total_segments * 100) if total_segments else 0.0

    print("=== Resume Summary ===")
    print(f"Range: {start_dt.date()} → {end_dt.date()} (end-exclusive)")
    print(f"Queries: {len(queries)} | Days: {len(day_keys)} | Sorts: {len(sorts)}")
    print(f"Segments total: {total_segments}")
    print(f"Done: {done_segments} | In-progress: {partial_segments} | Not-started: {not_started_segments}")
    print(f"Completion: {pct:.2f}%\n")

    print("=== Per-query progress ===")
    for q in queries:
        q_done, q_partial, q_not_started = q_stats[q]
        q_total = q_done + q_partial + q_not_started
        q_pct = (q_done / q_total * 100) if q_total else 0.0
        print(f"- {q!r}: {q_done}/{q_total} done ({q_pct:.1f}%), "
              f"{q_partial} in-progress, {q_not_started} not-started")
    print()

    print("=== Next work item per query ===")
    for q in queries:
        item = next_work.get(q)
        if not item:
            print(f"- {q!r}: [complete]")
            continue
        day, s, status, pages, cursor = item
        cursor_flag = "end" if not cursor else "has-cursor"
        print(f"- {q!r}: {day} | {s} | {status} | pages={pages} | cursor={cursor_flag}")
    print()

# ----------------------------------------------------------------------
# ----------------------------------------------------------------------
# Consolidated export

def export_consolidated(out_path: str = CONSOLIDATED_OUT) -> None:
 # writes a consolidated JSONL where each line is a post with all its matches embedded   
    posts = conn.execute("SELECT uri, json FROM posts_json").fetchall()

    # build a URI -> [matches] index in one pass over the matches table
    matches_by_uri: Dict[str, List[Dict[str, Any]]] = {}
    cur = conn.execute("""
        SELECT uri, matched_query, day_bucket, sort, createdAt, in_day_window
        FROM post_matches
        ORDER BY day_bucket, matched_query, sort
    """)
    for uri, q, day, sort, created, in_day in cur.fetchall():
        matches_by_uri.setdefault(uri, []).append({
            "matched_query": q,
            "day_bucket": day,
            "sort": sort,
            "createdAt": created,
            "in_day_window": bool(in_day),
        })

    with open(out_path, "w", encoding="utf-8") as f:
        for uri, post_json in tqdm(posts, desc="Export consolidated"):
            post_obj = json.loads(post_json)
            post_obj["matches"] = matches_by_uri.get(uri, [])
            f.write(json.dumps(post_obj, ensure_ascii=False) + "\n")

    print(f"Wrote consolidated JSONL -> {out_path}")


In [ ]:
resume_summary(state, QUERIES, START_DT, END_DT)

In [ ]:
run_daily_fetch()